In [1]:
import sys
from pathlib import Path
import pandas as pd
from decimal import Decimal

# Add src to path
src_path = Path().absolute().parent.parent / 'src'
sys.path.insert(0, str(src_path))

from cryptoquant.collectors.coinbase_client import CoinbaseClient
from cryptoquant.database.session import get_session
from cryptoquant.database.models import Asset, TradingPair

In [2]:
# Initialize client and session
client = CoinbaseClient()
session = get_session()
print("✅ Database connection successful")

✅ Database connection successful


In [3]:
# Fetch all products from Coinbase API
products = client.get_products()
print(f"Fetched {len(products)} products from Coinbase API")

Fetched 912 products from Coinbase API


In [ ]:
# Convert to DataFrame for inspection
products_df = pd.DataFrame([p.model_dump() for p in products])
print(f"\nColumns: {list(products_df.columns)}")
products_df.head()

In [4]:
# Get existing assets to create a symbol -> id mapping
assets = session.query(Asset).all()
asset_map = {asset.symbol: asset.id for asset in assets}
print(f"Found {len(asset_map)} existing assets in database")
print(f"Sample assets: {list(asset_map.keys())[:10]}")

Found 409 existing assets in database
Sample assets: ['00', '1INCH', '2Z', 'A8', 'AAVE', 'ABT', 'ACH', 'ACS', 'ADA', 'AERGO']


In [5]:
# Check for missing assets (currencies not yet in Asset table)
all_currencies = set()
for product in products:
    all_currencies.add(product.base_currency_id)
    all_currencies.add(product.quote_currency_id)

missing_currencies = all_currencies - set(asset_map.keys())
print(f"\nFound {len(missing_currencies)} currencies not in Asset table:")
print(f"Missing: {sorted(list(missing_currencies))[:20]}")


Found 0 currencies not in Asset table:
Missing: []


In [6]:
# Insert missing assets
new_assets = []
for currency in missing_currencies:
    # Determine asset type (fiat vs crypto)
    fiat_currencies = {'USD', 'EUR', 'GBP', 'USDC', 'USDT', 'DAI', 'UST'}
    asset_type = 'fiat' if currency in fiat_currencies else 'cryptocurrency'
    
    new_asset = Asset(
        symbol=currency,
        name=currency,  # Will be updated later if needed
        display_symbol=currency,
        asset_type=asset_type,
        decimals=8,
        active=True
    )
    new_assets.append(new_asset)

if new_assets:
    session.add_all(new_assets)
    session.commit()
    print(f"\n✅ Inserted {len(new_assets)} new assets")
    
    # Refresh asset map
    assets = session.query(Asset).all()
    asset_map = {asset.symbol: asset.id for asset in assets}
    print(f"Total assets now: {len(asset_map)}")
else:
    print("\n✅ No new assets to insert")


✅ No new assets to insert


In [7]:
# Check for existing trading pairs to avoid duplicates
existing_pairs = session.query(TradingPair.symbol).all()
existing_pair_symbols = {pair.symbol for pair in existing_pairs}
print(f"Found {len(existing_pair_symbols)} existing trading pairs")

Found 0 existing trading pairs


In [8]:
# Prepare trading pairs for insertion
trading_pairs = []
skipped = 0

for product in products:
    # Skip if pair already exists
    if product.product_id in existing_pair_symbols:
        skipped += 1
        continue
    
    # Get base and quote asset IDs
    base_asset_id = asset_map.get(product.base_currency_id)
    quote_asset_id = asset_map.get(product.quote_currency_id)
    
    if not base_asset_id or not quote_asset_id:
        print(f"⚠️  Skipping {product.product_id}: Missing asset mapping")
        continue
    
    # Convert string values to Decimal for numeric fields
    trading_pair = TradingPair(
        base_asset_id=base_asset_id,
        quote_asset_id=quote_asset_id,
        symbol=product.product_id,
        status=product.status,
        trading_disabled=product.trading_disabled,
        active=not product.trading_disabled,
        base_increment=Decimal(product.base_increment) if product.base_increment else None,
        quote_increment=Decimal(product.quote_increment) if product.quote_increment else None,
        base_min_size=Decimal(product.base_min_size) if product.base_min_size else None,
        base_max_size=Decimal(product.base_max_size) if product.base_max_size else None,
        quote_min_size=Decimal(product.quote_min_size) if product.quote_min_size else None,
        quote_max_size=Decimal(product.quote_max_size) if product.quote_max_size else None
    )
    trading_pairs.append(trading_pair)

print(f"\nPrepared {len(trading_pairs)} trading pairs for insertion")
print(f"Skipped {skipped} existing pairs")


Prepared 912 trading pairs for insertion
Skipped 0 existing pairs


In [9]:
# Insert trading pairs in batches
batch_size = 100
total = len(trading_pairs)

for i in range(0, total, batch_size):
    batch = trading_pairs[i:i+batch_size]
    session.add_all(batch)
    session.commit()
    print(f"Inserted batch {i//batch_size + 1}: {len(batch)} pairs")

print(f"\n✅ Successfully inserted {total} trading pairs")

Inserted batch 1: 100 pairs


DataError: (pyodbc.DataError) ('22003', '[22003] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Arithmetic overflow error converting numeric to data type numeric. (8115) (SQLExecDirectW); [22003] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)')
[SQL: INSERT INTO crypto.trading_pairs (base_asset_id, quote_asset_id, symbol, status, trading_disabled, active, base_increment, quote_increment, base_min_size, base_max_size, quote_min_size, quote_max_size) OUTPUT inserted.id, inserted.created_at, inserte ... 4288 characters truncated ...  99)) AS imp_sen(p0, p1, p2, p3, p4, p5, p6, p7, p8, p9, p10, p11, sen_counter) ORDER BY sen_counter]
[parameters: (1524, 1592, 'RED-USD', 'online', 0, 1, Decimal('0.01'), Decimal('0.0001'), Decimal('0.01'), Decimal('20201650.6342478415528703'), Decimal('1'), Decimal('10000000'), 1524, 1594, 'RED-USDC', 'online', 0, 1, Decimal('0.01'), Decimal('0.0001'), Decimal('0.01'), Decimal('20201650.6342478415528703

In [ ]:
# Verify insertion
total_pairs = session.query(TradingPair).count()
print(f"\nTotal trading pairs in database: {total_pairs}")

# Show sample pairs
sample_pairs = session.query(TradingPair).limit(10).all()
print("\nSample trading pairs:")
for pair in sample_pairs:
    base_asset = session.query(Asset).filter(Asset.id == pair.base_asset_id).first()
    quote_asset = session.query(Asset).filter(Asset.id == pair.quote_asset_id).first()
    print(f"  {pair.symbol}: {base_asset.symbol}/{quote_asset.symbol} | Status: {pair.status} | Trading: {'✓' if not pair.trading_disabled else '✗'}")

In [10]:
# Check tracked pairs are present
tracked_pair_ids = ['BTC-USD', 'ETH-USD', 'XRP-USD', 'SOL-USD']
tracked_pairs = session.query(TradingPair).filter(TradingPair.symbol.in_(tracked_pair_ids)).all()

print(f"\n✅ Tracked pairs verification:")
for pair in tracked_pairs:
    print(f"  {pair.symbol} (ID: {pair.id}) | Status: {pair.status} | Base min: {pair.base_min_size}")

PendingRollbackError: This Session's transaction has been rolled back due to a previous exception during flush. To begin a new transaction with this Session, first issue Session.rollback(). Original exception was: (pyodbc.DataError) ('22003', '[22003] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Arithmetic overflow error converting numeric to data type numeric. (8115) (SQLExecDirectW); [22003] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)')
[SQL: INSERT INTO crypto.trading_pairs (base_asset_id, quote_asset_id, symbol, status, trading_disabled, active, base_increment, quote_increment, base_min_size, base_max_size, quote_min_size, quote_max_size) OUTPUT inserted.id, inserted.created_at, inserte ... 4288 characters truncated ...  99)) AS imp_sen(p0, p1, p2, p3, p4, p5, p6, p7, p8, p9, p10, p11, sen_counter) ORDER BY sen_counter]
[parameters: (1524, 1592, 'RED-USD', 'online', 0, 1, Decimal('0.01'), Decimal('0.0001'), Decimal('0.01'

In [ ]:
session.close()
print("\n✅ Session closed")